# Apache Spark Data Lake Preprocessing
## 1. Data Lake Structure Setup & PySpark Installation

In [1]:
!pip install pyspark
import os

os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)
print("Data Lake folders created: data_lake/bronze, data_lake/silver/preprocessed_sales")

Data Lake folders created: data_lake/bronze, data_lake/silver/preprocessed_sales


**🛑 IMPORTANT**: Before proceeding, please upload your `sales.csv` dataset into the `data_lake/bronze/` folder using the Colab file browser on the left.

## 2. Load and Inspect with Spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, when, count, isnull, to_date

spark = SparkSession.builder.appName("DataLakePreprocessing").getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("--- Schema ---")
df.printSchema()

print("--- First 5 Rows ---")
df.show(5)

initial_count = df.count()
print(f"Total Record Count: {initial_count}")

--- Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

--- First 5 Rows ---
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+------

In [3]:
duplicate_count = df.count() - df.dropDuplicates().count()
print(f"Exact Duplicate Rows: {duplicate_count}")

print("Null Values per Column:")
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show()

Exact Duplicate Rows: 10
Null Values per Column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



## 3. Basic Preprocessing - Silver Layer
Here we will perform basic cleaning: removing duplicates, standardizing text, handling missing/invalid values, and parsing dates.

In [5]:
from pyspark.sql.functions import expr

df_silver = df.dropDuplicates()

cols = [c.lower() for c in df_silver.columns]

for c in df_silver.columns:
    c_lower = c.lower()

    if dict(df_silver.dtypes)[c] == 'string':
        df_silver = df_silver.withColumn(c, initcap(trim(col(c))))

    if 'quantity' in c_lower:
        df_silver = df_silver.na.fill(0, subset=[c])
        df_silver = df_silver.withColumn(c, when(col(c) < 0, 0).otherwise(col(c)))

    if 'price' in c_lower:
        df_silver = df_silver.na.fill(0.0, subset=[c])
        df_silver = df_silver.withColumn(c, when(col(c) < 0.0, 0.0).otherwise(col(c)))

    if 'discount' in c_lower:
        df_silver = df_silver.na.fill(0.0, subset=[c])
        df_silver = df_silver.withColumn(c, when((col(c) < 0) | (col(c) > 100), 0.0).otherwise(col(c)))

    if 'date' in c_lower:
        df_silver = df_silver.withColumn(c, expr(f"try_cast({c} as date)"))

silver_path = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.mode("overwrite").csv(silver_path, header=True)
print(f"Processed data saved to {silver_path}")


Processed data saved to data_lake/silver/preprocessed_sales


## 4. Verification

In [6]:
final_count = df_silver.count()
print(f"Row count BEFORE preprocessing: {initial_count}")
print(f"Row count AFTER preprocessing: {final_count}")

# Read the Silver data back and display a few rows
df_silver_read = spark.read.csv(silver_path, header=True, inferSchema=True)
print("\n--- Silver Layer Data (First 5 Rows) ---")
df_silver_read.show(5)

Row count BEFORE preprocessing: 1000
Row count AFTER preprocessing: 990

--- Silver Layer Data (First 5 Rows) ---
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|     670.0|            30.0|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|     670.0|             0.0|     Net Banking|      Pune|Maharashtra|2025-05-08|   Delivered|
|  100506|      

### Preprocessing Summary
1. **Duplicates**: Dropped exact duplicate rows.
2. **Text Standardization**: Applied `trim()` to remove whitespace and `initcap()` to standardize capitalization for all string columns.
3. **Numerical Validation**: Replaced negative quantities/prices with 0. Ensured discounts stay within logical bounds. Null values in numerical columns were filled with 0.
4. **Dates**: Converted date columns to a standard format using `to_date()`.